# Student Wellbeing Survey — Statistical Analysis

Download the Student Wellbeing Survey CSV from the LMS **Study Material** tab. Run the one code cell below in Google Colab, then upload the file when prompted.

In [ ]:
# Student Wellbeing Survey statistical analysis — one Google Colab cell
import pandas as pd
import numpy as np
from google.colab import files

uploaded = files.upload()
csv_files = [name for name in uploaded if name.lower().endswith('.csv')]
if not csv_files:
    raise FileNotFoundError('Please upload the Student Wellbeing Survey CSV file.')

df = pd.read_csv(csv_files[0])
df.columns = df.columns.str.strip().str.replace(r'\s+', '_', regex=True)
pd.set_option('display.max_columns', None)

def col(name):
    matches = [c for c in df.columns if c.lower() == name.lower()]
    if not matches:
        raise KeyError(f'Required column not found: {name}. Available columns: {df.columns.tolist()}')
    return matches[0]

study = col('Weekly_Study_Hours')
sleep = col('Average_Sleep_Hours')
screen = col('Daily_Screen_Time_Hours')
stress = col('Stress_Score')
readiness = col('Academic_Readiness_Score')
commute = col('Commute_Time_Minutes')
spending = col('Monthly_Discretionary_Spending')
job = col('Part_Time_Job')
scholarship = col('Scholarship')
exercise = col('Exercise_Days_Per_Week')
year = col('Year_of_Study')
variables = [study, sleep, screen, stress, readiness]

for variable in variables + [commute, spending, exercise, year]:
    df[variable] = pd.to_numeric(df[variable], errors='coerce')

print('=' * 95)
print(f'STUDENT WELLBEING SURVEY — STATISTICAL ANALYSIS: {csv_files[0]}')
print('=' * 95)
print(f'Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
display(df.head())
print('\nData types and missing values:')
display(pd.DataFrame({'Data Type': df.dtypes.astype(str), 'Missing Values': df.isnull().sum()}))

# 1. CENTRAL TENDENCY
print('\n1. MEAN, MEDIAN, AND MODE')
print('Formulas: Mean = Σx/n | Median = middle ordered value | Mode = most frequent value')
central_rows = []
for variable in variables:
    values = df[variable].dropna()
    modes = values.mode().tolist()
    central_rows.append([variable, values.mean(), values.median(), ', '.join(f'{m:g}' for m in modes)])
central = pd.DataFrame(central_rows, columns=['Variable', 'Mean', 'Median', 'Mode(s)'])
central[['Mean', 'Median']] = central[['Mean', 'Median']].round(2)
display(central)

# 2. DISPERSION
print('\n2. RANGE, VARIANCE, STANDARD DEVIATION, QUARTILES, AND IQR')
print('Formulas: Range = max − min | Variance = Σ(x − x̄)²/(n−1) | SD = √Variance | IQR = Q3 − Q1')
dispersion_rows = []
for variable in variables:
    values = df[variable].dropna()
    q1, q3 = values.quantile(0.25), values.quantile(0.75)
    dispersion_rows.append([variable, values.max() - values.min(), values.var(ddof=1), values.std(ddof=1), q1, q3, q3 - q1])
dispersion = pd.DataFrame(dispersion_rows, columns=['Variable', 'Range', 'Variance', 'Standard Deviation', 'Q1', 'Q3', 'IQR']).round(2)
display(dispersion)
most_variable = dispersion.loc[dispersion['Standard Deviation'].idxmax()]
print(f'Interpretation: {most_variable["Variable"]} has the greatest variability by standard deviation ({most_variable["Standard Deviation"]:.2f}) among these variables.')

# 3. IQR OUTLIER DETECTION
print('\n3. IQR OUTLIER ANALYSIS')
print('Formula: Lower bound = Q1 − 1.5×IQR; Upper bound = Q3 + 1.5×IQR. Values outside these bounds are outliers.')
outlier_variables = [study, screen, commute, spending]
outlier_rows, outlier_masks = [], {}
for variable in outlier_variables:
    q1, q3 = df[variable].quantile(0.25), df[variable].quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    mask = (df[variable] < lower) | (df[variable] > upper)
    outlier_masks[variable] = mask
    outlier_rows.append([variable, q1, q3, iqr, lower, upper, int(mask.sum()), round(mask.mean() * 100, 2)])
outlier_table = pd.DataFrame(outlier_rows, columns=['Variable', 'Q1', 'Q3', 'IQR', 'Lower Bound', 'Upper Bound', 'Outliers', 'Outliers (%)']).round(2)
display(outlier_table)

# Compare daily screen time before and after removing its IQR-detected outliers.
screen_without_outliers = df.loc[~outlier_masks[screen], screen].dropna()
comparison = pd.DataFrame({
    'Measure': ['Mean', 'Median', 'Number of observations'],
    'Before removing screen-time outliers': [df[screen].mean(), df[screen].median(), df[screen].notna().sum()],
    'After removing screen-time outliers': [screen_without_outliers.mean(), screen_without_outliers.median(), screen_without_outliers.shape[0]]
})
comparison.iloc[:2, 1:] = comparison.iloc[:2, 1:].astype(float).round(2)
display(comparison)
print(f'Interpretation: removing {outlier_masks[screen].sum()} detected Daily_Screen_Time_Hours outlier(s) changes the mean from {df[screen].mean():.2f} to {screen_without_outliers.mean():.2f} and the median from {df[screen].median():.2f} to {screen_without_outliers.median():.2f}.')

# 4. PROBABILITY
print('\n4. PROBABILITY ANALYSIS')
print('Definitions: A = Part_Time_Job = Yes; B = Stress_Score ≥ 7; C = Scholarship = Yes; D = Exercise_Days_Per_Week ≥ 3')
A = df[job].astype(str).str.strip().str.lower().eq('yes')
B = df[stress].ge(7)
C = df[scholarship].astype(str).str.strip().str.lower().eq('yes')
D = df[exercise].ge(3)
n = len(df)
P_A, P_B, P_C, P_D = A.mean(), B.mean(), C.mean(), D.mean()
P_A_or_B, P_A_and_B = (A | B).mean(), (A & B).mean()
P_A_given_B = P_A_and_B / P_B if P_B else np.nan
P_B_given_A = P_A_and_B / P_A if P_A else np.nan
probabilities = pd.DataFrame({
    'Probability': ['P(A)', 'P(B)', 'P(C)', 'P(D)', 'P(A or B)', 'P(A and B)', 'P(A|B)', 'P(B|A)'],
    'Formula': ['count(A)/n', 'count(B)/n', 'count(C)/n', 'count(D)/n', 'P(A)+P(B)−P(A and B)', 'count(A and B)/n', 'P(A and B)/P(B)', 'P(A and B)/P(A)'],
    'Value': [P_A, P_B, P_C, P_D, P_A_or_B, P_A_and_B, P_A_given_B, P_B_given_A]
})
probabilities['Value'] = probabilities['Value'].map(lambda x: f'{x:.4f} ({x*100:.2f}%)' if pd.notna(x) else 'Undefined')
display(probabilities)
year_1, year_4 = df[year].eq(1), df[year].eq(4)
intersection_years = int((year_1 & year_4).sum())
print(f'Mutual exclusivity: P(Year 1 and Year 4) = {intersection_years}/{n} = 0. Therefore, Year_of_Study = 1 and Year_of_Study = 4 are mutually exclusive events.')
product_independent = P_A * P_B
difference = abs(P_A_and_B - product_independent)
independence_conclusion = 'appear independent (the values are very close)' if difference < 0.01 else 'do not appear independent (the values differ materially)'
print(f'Independence check: P(A and B) = {P_A_and_B:.4f}; P(A)×P(B) = {product_independent:.4f}; difference = {difference:.4f}. A and B {independence_conclusion}.')

# 5. BAYES' THEOREM
print('\n5. BAYES THEOREM')
not_A = ~A
P_B_given_not_A = B[not_A].mean() if not_A.any() else np.nan
denominator = P_B_given_A * P_A + P_B_given_not_A * (1 - P_A)
bayes_A_given_B = (P_B_given_A * P_A) / denominator if denominator else np.nan
print('Formula: P(A|B) = [P(B|A) × P(A)] / [P(B|A) × P(A) + P(B|not A) × P(not A)]')
print(f'P(B|A) = {P_B_given_A:.4f}; P(B|not A) = {P_B_given_not_A:.4f}; P(A) = {P_A:.4f}')
print(f'Bayes result P(A|B) = {bayes_A_given_B:.4f} ({bayes_A_given_B*100:.2f}%)')
print(f'Direct result P(A|B) = {P_A_given_B:.4f} ({P_A_given_B*100:.2f}%)')
print(f'Verification: the difference is {abs(bayes_A_given_B - P_A_given_B):.10f}; the two calculations agree up to rounding.')

# 6. NORMAL DISTRIBUTION AND Z-SCORES
print('\n6. NORMAL DISTRIBUTION: ACADEMIC READINESS SCORE')
readiness_values = df[readiness].dropna()
readiness_mean, readiness_sd = readiness_values.mean(), readiness_values.std(ddof=1)
highest, lowest = readiness_values.max(), readiness_values.min()
z_high, z_low = (highest - readiness_mean) / readiness_sd, (lowest - readiness_mean) / readiness_sd
print('Formula: Z = (x − μ) / σ')
print(f'Mean (μ) = {readiness_mean:.2f}; Standard deviation (σ) = {readiness_sd:.2f}')
print(f'Highest score = {highest:.2f}; Z-score = ({highest:.2f} − {readiness_mean:.2f}) / {readiness_sd:.2f} = {z_high:.2f}')
print(f'Interpretation: the highest-score student is {z_high:.2f} standard deviations above the mean.')
print(f'Lowest score = {lowest:.2f}; Z-score = ({lowest:.2f} − {readiness_mean:.2f}) / {readiness_sd:.2f} = {z_low:.2f}')
print(f'Interpretation: the lowest-score student is {abs(z_low):.2f} standard deviations below the mean.')
empirical = pd.DataFrame({
    'Standard-deviation interval': ['Within μ ± 1σ', 'Within μ ± 2σ', 'Within μ ± 3σ'],
    'Expected percentage (68-95-99.7 rule)': ['68%', '95%', '99.7%'],
    'Observed percentage in this dataset': [
        f'{((readiness_values.between(readiness_mean-readiness_sd, readiness_mean+readiness_sd)).mean()*100):.2f}%',
        f'{((readiness_values.between(readiness_mean-2*readiness_sd, readiness_mean+2*readiness_sd)).mean()*100):.2f}%',
        f'{((readiness_values.between(readiness_mean-3*readiness_sd, readiness_mean+3*readiness_sd)).mean()*100):.2f}%'
    ]
})
display(empirical)

# 7. FIVE DATA-DRIVEN OBSERVATIONS
print('\n7. FIVE STATISTICAL OBSERVATIONS')
observations = [
    f'1. {most_variable["Variable"]} has the largest standard deviation ({most_variable["Standard Deviation"]:.2f}), making it the most variable of the five core wellbeing measures.',
    f'2. Daily screen time has {int(outlier_masks[screen].sum())} IQR-detected outlier(s); removing them changes its mean by {abs(df[screen].mean() - screen_without_outliers.mean()):.2f} hours.',
    f'3. The probability of a student having a part-time job is {P_A*100:.2f}%, while the probability of high stress (score ≥ 7) is {P_B*100:.2f}%.',
    f'4. The part-time-job and high-stress events {independence_conclusion}; P(A and B) is {P_A_and_B:.4f} compared with P(A)×P(B) of {product_independent:.4f}.',
    f'5. The highest Academic_Readiness_Score is {z_high:.2f} standard deviations above the mean, while the lowest is {abs(z_low):.2f} standard deviations below it.'
]
for observation in observations:
    print(observation)